## Initialization

In [1]:
from active_learning.al_new import AL
from utils.utils import get_tasks

# exp selection
# exp_name = 'acidic_mor_PtRuSc'
# exp_name = 'MOR_5D'
# exp_name = 'DFFC_PdPtCu'
# exp_name = 'DFFC_8D_4'
exp_name = 'Thomas_battery_PBSi'

## Save and load

In [2]:
# load al instance from database
al = AL(exp_name)

In [6]:
# save the current al instance to the database
al.save_exp_to_db()

## AL main workflow

### recipe generation

In [9]:
%%time
# generate next trial using Bayesian Optimization, specify beta if needed, e.g. beta=100 for exploration, beta=0.2 for default
al.generate_botorch_trial(n=10, beta=0, aquisition = 'qUCB', optimizer_kwargs = {'options':{'maxiter':20000000}, 'sequential':True})

BO trial successfully added as trial 4
CPU times: total: 1min 55s
Wall time: 1min 6s


In [ ]:
# generate next trial using SOBOL method (random select)
al.generate_sobol_trial(n=20)

In [7]:
# update the recipe table on the database, if trial index is not specified, the recipe of the latest trial will be uploaded
al.update_recipe_table(trial_index=None, add_pred=True)

### sample preparation

In [3]:
# benchmark specification
benchmark = None
# benchmark={5: '1_0'}

In [ ]:
# command opentrons to prepare the sample for the latest trial, following the ot2_run_configs of the exp
al.make_sample(trial_index=None, benchmark=benchmark, operator_name='chu')

In [4]:
# update the sample table on the database, if trial index is not specified, the sample of the latest trial will be uploaded. Note that sample table calculation is based on local ot2_run_configs.py under project folder, not the one on ot2 server.
al.update_sample_table(trial_index=None, benchmark=benchmark)

### sample testing complete

In [5]:
# run this cell after sample testing and data analysis is completed, then go back to the BO step
al.mark_trial_complete(trial_index=None)

BatchTrial(experiment_name='DFFC_8D_4', index=13, status=TrialStatus.RUNNING)


## AL monitor

In [ ]:
# update the exp with the latest result
al.update_exp_result()

In [11]:
# exp monitor for windows
al.exp_monitor()

,arm_name,trial_index,arm_index,u,v,x,pred_mean,pred_std,_10th_DCh,trial_status,generation_method
0,0_0,0,0,0.150,0.150,0.150,886.8,14.18,900.0,COMPLETED,Manual
1,1_0,1,0,0.210,0.130,0.180,883.2,14.18,870.0,COMPLETED,Manual
2,2_0,2,0,0.000,0.562,0.000,890.4,34.51,NaN,RUNNING,BoTorch
3,2_1,2,1,0.000,0.000,0.592,884.6,38.84,NaN,RUNNING,BoTorch
4,2_1,2,1,0.000,0.000,0.592,884.6,38.84,NaN,RUNNING,BoTorch
21,2_10,2,10,0.213,0.293,0.000,887.0,26.74,NaN,RUNNING,BoTorch
22,2_11,2,11,0.160,0.027,0.000,888.1,25.86,NaN,RUNNING,BoTorch
23,2_12,2,12,0.000,0.181,0.456,885.6,35.76,NaN,RUNNING,BoTorch
24,2_13,2,13,0.000,0.627,0.122,889.7,33.90,NaN,RUNNING,BoTorch
25,2_14,2,14,0.141,0.000,0.416,882.7,32.05,NaN,RUNNING,BoTorch


In [ ]:
# exp monitor for mac and linux
from IPython.display import display, HTML
display(HTML(al.exp_monitor().to_html()))

In [ ]:
# check abandon reason
al.exp.trials[0].abandoned_reason

## AL manual edit

In [ ]:
# update the exp with the latest result and force match previous trials with the results in the database
al.update_exp_result(force_update_trial=[0, 1])

In [ ]:
# manually add a trial
trial_df = get_tasks(exp_name, task_name='manual')
al.add_manual_trial(trial_df)

In [ ]:
# abandon an certain arm in a trial
al.abandon_arm(0, '0_18', 'abandoned in manual test')

In [ ]:
# abandon a whole trial
al.abandon_trial(9, 'test trial')

## Plotting

In [10]:
from ax.modelbridge.cross_validation import cross_validate
from ax.plot.contour import interact_contour
from ax.plot.diagnostic import interact_cross_validation
from ax.plot.scatter import interact_fitted
from ax.plot.slice import interact_slice
from ax.utils.notebook.plotting import render, init_notebook_plotting
init_notebook_plotting()

[INFO 10-18 10:00:25] ax.utils.notebook.plotting: Injecting Plotly library into cell. Do not overwrite or delete cell.


In [11]:
model = al.get_bo_model()

In [ ]:
# Contour plots
render(interact_contour(model=model, metric_name='max_power'))

In [12]:
# Cross-validation plots
cv_results = cross_validate(model)
print('current r_squared is: ', al.calculate_r_squared(cv_results))
render(interact_cross_validation(cv_results))

current r_squared is:  0.8406


In [ ]:
# Slice plots
render(interact_slice(model))

In [ ]:
# Tile plots
render(interact_fitted(model, rel=False))

## Danger Zone

In [ ]:
# delete the current active learning process from the database
al.delete_exp_on_db()

In [ ]:
# only use this if you want to start a new experiment, note that the previous experiment has to be manually deleted, first, because the script refuses to overwrite the previous experiment by default
al = AL(exp_name, load_from_db=False)